In [5]:

import os
import random
import re
from collections import defaultdict
from xml.etree.ElementTree import tostring

from PIL import Image
import numpy as np
from pathlib import Path

class DatasetGenerator:
    def __init__(self, background_dir, ground_units_dir, air_units_dir, towers_dir, misc_dir, output_dir):
        """
        Inizializza il generatore di dataset

        Args:
            background_dir: cartella con le immagini di background
            ground_units_dir: cartella con PNG delle truppe di terra
            air_units_dir: cartella con PNG delle truppe aeree
            towers_dir: cartella con PNG delle torri
            output_dir: cartella di output per immagini e labels
        """
        self.background_dir = Path(background_dir)
        self.ground_units_dir = Path(ground_units_dir)
        self.air_units_dir = Path(air_units_dir)
        self.towers_dir = Path(towers_dir)
        self.misc_dir = Path(misc_dir)
        self.output_dir = Path(output_dir)
        # Crea le cartelle di output
        (self.output_dir / "images").mkdir(parents=True, exist_ok=True)
        (self.output_dir / "labels").mkdir(parents=True, exist_ok=True)

        # Carica le liste dei file
        self.backgrounds = list(self.background_dir.glob("*.jpg")) + list(self.background_dir.glob("*.png"))
        self.ground_units = list(self.ground_units_dir.glob("*.png"))
        self.air_units = list(self.air_units_dir.glob("*.png"))
        self.towers = list(self.towers_dir.glob("*.png"))
        self.misc = list(self.misc_dir.glob("*.png"))
        nomi = []
        for x in self.ground_units:
            nomi.append(os.path.basename(x))
        self.bucketsGround = defaultdict(list)
        for f in nomi:
            # prende la parte tra _ e ( oppure _ e .png
            match = re.search(r'_(.*?)(?:\s*\(|\.png)', f)
            if match:
                key = match.group(1)
                self.bucketsGround[key].append(f)
        nomi=[]
        for x in self.air_units:
            nomi.append(os.path.basename(x))
        self.bucketsAir = defaultdict(list)

        for f in nomi:
            # prende la parte tra _ e ( oppure _ e .png
            match = re.search(r'_(.*?)(?:\s*\(|\.png)', f)
            if match:
                key = match.group(1)
                self.bucketsAir[key].append(f)

        # Mappa delle classi (modifica secondo le tue esigenze)
        self.class_map = {}
        n=0
        for k in self.bucketsGround.keys():
            self.class_map[k] = n
            n += 1
        for k in self.bucketsAir.keys():
            self.class_map[k] = n
            n += 1
        self.class_map["tower"] = n
        base_dir = Path('imported_segments/ground_units')

        self.bucketsGround= {
            key: [base_dir / filename for filename in files]
            for key, files in self.bucketsGround.items()
        }
        base_dir = Path('imported_segments/air_units')

        self.bucketsAir= {
            key: [base_dir / filename for filename in files]
            for key, files in self.bucketsAir.items()
        }




    def load_png_with_alpha(self, path):
        """Carica un PNG preservando il canale alpha"""
        img = Image.open(path).convert("RGBA")
        try:
            nome = re.search(r"_([^\s.(]+)", os.path.basename(path))
            nome=nome.group(1)
        except Exception as e:
            return img
        return img , nome

    def paste_with_transparency(self, background, overlay, position):
        """Incolla un'immagine con trasparenza su un background"""
        background.paste(overlay, position, overlay)
        return background

    def get_bbox_yolo(self, x, y, width, height, img_width, img_height):
        """
        Converte coordinate bbox in formato YOLO
        YOLO formato: [class_id, x_center, y_center, width, height] (normalizzato 0-1)
        """
        x_center = (x + width / 2) / img_width
        y_center = (y + height / 2) / img_height
        norm_width = width / img_width
        norm_height = height / img_height
        return [x_center, y_center, norm_width, norm_height]

    def generate_image(self, num_ground_units, num_air_units):
        """
        Genera una singola immagine con annotazioni YOLO

        Returns:
            image: immagine PIL generata
            annotations: lista di annotazioni YOLO [class_id, x_center, y_center, width, height]
        """
        # Carica background casuale
        bg_path = random.choice(self.backgrounds)
        background = Image.open(bg_path).convert("RGBA")
        img_width, img_height = background.size
        annotations = []

        # Fattori di scala
        # Da 442x652 a 568x896
        scale_x = img_width / 442
        scale_y = img_height / 652

        # Exclusions scalate
        exclusions = [
            (int(60*scale_x), int(100*scale_y), int(120*scale_x), int(170*scale_y)),
            (int(180*scale_x), int(10*scale_y), int(250*scale_x), int(120*scale_y)),
            (int(320*scale_x), int(100*scale_y), int(380*scale_x), int(170*scale_y)),
            (int(60*scale_x), int(470*scale_y), int(120*scale_x), int(540*scale_y)),
            (int(180*scale_x), int(520*scale_y), int(250*scale_x), int(610*scale_y)),
            (int(320*scale_x), int(470*scale_y), int(380*scale_x), int(540*scale_y))
        ]

        # Livello 2: Truppe di terra e torri
        # Aggiungi torri
        x = int(50 * scale_x)
        y = int(460 * scale_y)
        ally_princess_tower, a = self.load_png_with_alpha(self.towers[1])
        ally_king_tower, a = self.load_png_with_alpha(self.towers[0])
        enemy_princess_tower, a = self.load_png_with_alpha(self.towers[3])
        enemy_king_tower, a = self.load_png_with_alpha(self.towers[2])

        # Torri alleate (basso)
        for _ in range(2):
            if not self.towers:
                continue

            tw, th = ally_princess_tower.size
            background = self.paste_with_transparency(background, ally_princess_tower, (x, y))
            # Calcola bbox YOLO
            bbox = self.get_bbox_yolo(x, y, tw, th, img_width, img_height)
            annotations.append([self.class_map['tower']] + bbox)
            x = int(315 * scale_x)

        # Torri nemiche (alto)
        x = int(55 * scale_x)
        y = int(100 * scale_y)
        for _ in range(2):
            if not self.towers:
                continue

            tw, th = enemy_princess_tower.size
            background = self.paste_with_transparency(background, enemy_princess_tower, (x, y))
            # Calcola bbox YOLO
            bbox = self.get_bbox_yolo(x, y, tw, th, img_width, img_height)
            annotations.append([self.class_map['tower']] + bbox)
            x = int(320 * scale_x)

        # Torre re nemica (alto centro)
        x = int(180 * scale_x)
        y = int(22 * scale_y)
        tw, th = enemy_king_tower.size
        background = self.paste_with_transparency(background, enemy_king_tower, (x, y))
        # Calcola bbox YOLO
        bbox = self.get_bbox_yolo(x, y, tw, th, img_width, img_height)
        annotations.append([self.class_map['tower']] + bbox)

        # Torre re alleata (basso centro)
        y = int(525 * scale_y)
        tw, th = ally_king_tower.size
        background = self.paste_with_transparency(background, ally_king_tower, (x, y))
        # Calcola bbox YOLO
        bbox = self.get_bbox_yolo(x, y, tw, th, img_width, img_height)
        annotations.append([self.class_map["tower"]] + bbox)

        # Aggiungi truppe di terra
        for _ in range(num_ground_units):
            if not self.ground_units:
                continue
            unit, nome = self.load_png_with_alpha(random.choice(self.bucketsGround[self.selectorGround.get_key()]))
            uw, uh = unit.size
            x, y = get_random_position_with_exclusions(img_width, img_height, uw, uh, exclusions)
            background = self.paste_with_transparency(background, unit, (x, y))
            bbox = self.get_bbox_yolo(x, y, uw, uh, img_width, img_height)
            annotations.append([self.class_map[nome]] + bbox)

        # Livello 3: Truppe aeree
        for _ in range(num_air_units):
            if not self.air_units:
                continue
            air, nome = self.load_png_with_alpha(random.choice(self.bucketsAir[self.selectorAir.get_key()]))
            aw, ah = air.size

            x = random.randint(0, max(0, img_width - aw))
            y = random.randint(0, max(0, img_height - ah))

            background = self.paste_with_transparency(background, air, (x, y))

            bbox = self.get_bbox_yolo(x, y, aw, ah, img_width, img_height)
            annotations.append([self.class_map[nome]] + bbox)

        # Elementi vari
        for _ in range(random.randint(0, 5)):
            misc = self.load_png_with_alpha(random.choice(self.misc))
            aw, ah = misc.size
            x = random.randint(0, max(0, img_width - aw))
            y = random.randint(0, max(0, img_height - ah))
            background = self.paste_with_transparency(background, misc, (x, y))

        return background, annotations


    def generate_dataset(self, num_images, distribution=None):
        """
        Genera un dataset bilanciato

        Args:
            num_images: numero totale di immagini da generare
            distribution: dizionario con la distribuzione desiderata per ogni configurazione
                         Se None, usa distribuzione uniforme bilanciata
        """

        self.selectorGround = BalancedKeySelector(self.bucketsGround)
        self.selectorAir = BalancedKeySelector(self.bucketsAir)
        for i in range(num_images):
            # Scegli una configurazione casuale

            num_ground = random.randint(0,10)
            num_air = random.randint(0,4)

            # Genera immagine
            image, annotations = self.generate_image(num_ground, num_air)

            # Salva immagine
            img_name = f"image_{i:05d}.jpg"
            image_path = self.output_dir / "images" / img_name
            image.convert("RGB").save(image_path, "JPEG")

            # Salva annotazioni YOLO
            label_name = f"image_{i:05d}.txt"
            label_path = self.output_dir / "labels" / label_name

            with open(label_path, 'w') as f:
                for ann in annotations:
                    class_id = ann[0]
                    bbox = ann[1:]
                    f.write(f"{class_id} {' '.join(map(str, bbox))}\n")

            if (i + 1) % 100 == 0:
                print(f"Generate {i + 1}/{num_images} immagini")

        print(f"Dataset completato! {num_images} immagini generate.")
        self.create_yaml_config()

    def create_yaml_config(self):
        """Crea il file di configurazione YAML per YOLO"""
        names = list(self.class_map.keys())

        print(f"names: {names}")
        print(f"nc: {len(names)}  # number of classes")
        print(self.class_map)
        yaml_content = f"""# Dataset configuration for YOLO
train: train
val: valid

# Classes
names:
    {names}
nc: {len(names)}  # number of classes
"""
        yaml_path = self.output_dir / "dataset.yaml"
        with open(yaml_path, 'w') as f:
            f.write(yaml_content)
        print(f"File di configurazione salvato in: {yaml_path}")


# Funzione di test e verifica
def check_directories(bg_dir, ground_dir, air_dir, tower_dir):
    """Verifica che le cartelle esistano e contengano file"""
    print("=== VERIFICA CARTELLE ===")

    dirs = {
        'Backgrounds': bg_dir,
        'Ground Units': ground_dir,
        'Air Units': air_dir,
        'Towers': tower_dir
    }

    all_ok = True
    for name, path in dirs.items():
        p = Path(path)
        if not p.exists():
            print(f"❌ {name}: cartella '{path}' NON ESISTE")
            all_ok = False
        else:
            files = list(p.glob("*.png")) + list(p.glob("*.jpg"))
            print(f"✓ {name}: {len(files)} file trovati in '{path}'")
            if len(files) == 0:
                all_ok = False

    return all_ok


In [6]:
class BalancedKeySelector:
    def __init__(self, dizionario):
        self.keys = list(dizionario.keys())
        self.n_keys = len(self.keys)
        self.current_batch = []
        self.batch_index = 0

    def _next_batch(self):
        """Crea nuovo batch mischiato"""
        self.current_batch = self.keys.copy()
        random.shuffle(self.current_batch)
        self.batch_index = 0

    def get_key(self):
        """Restituisce 1 chiave bilanciata in ordine random"""
        if self.n_keys == 0:
            return None

        # Se finito il batch, creane uno nuovo
        if self.batch_index >= len(self.current_batch):
            self._next_batch()

        key = self.current_batch[self.batch_index]
        self.batch_index += 1
        return key

In [7]:
def get_random_position_with_exclusions(img_width, img_height, uw, uh, exclusions):
    """
    Trova posizione random escludendo zone specifiche.

    Args:
        img_width, img_height: dimensioni immagine
        uw, uh: dimensioni dell'unità da piazzare
        exclusions: lista di tuple (x, y, w, h) delle zone da evitare

    Returns:
        (x, y) posizione valida
    """
    max_attempts = 10

    for _ in range(max_attempts):
        x = random.randint(0, max(0, img_width - uw))
        y = random.randint(0, max(0, img_height - uh))

        # Controlla se si sovrappone con zone escluse
        valid = True
        for ex_x, ex_y, ex_w, ex_h in exclusions:
            # Controlla sovrapposizione
            if not (x + uw <= ex_x or  # completamente a sinistra
                    x >= ex_x + ex_w or  # completamente a destra
                    y + uh <= ex_y or    # completamente sopra
                    y >= ex_y + ex_h):   # completamente sotto
                valid = False
                break

        if valid:
            return x, y

    # Se non trova posizione valida, ritorna random comunque
    return random.randint(0, max(0, img_width - uw)), \
           random.randint(0, max(0, img_height - uh))




In [10]:
# Esempio di utilizzo
if __name__ == "__main__":
    generator = DatasetGenerator(
        background_dir="imported_segments/backgrounds",      # Cartella con i background
        ground_units_dir="imported_segments/ground_units",   # Cartella con truppe di terra PNG
        air_units_dir="imported_segments/air_units",         # Cartella con truppe aeree PNG
        towers_dir="imported_segments/towers",               # Cartella con torri PNG
        misc_dir="imported_segments/misc",           #cartella dei miscellaneous
        output_dir="dataset_output"        # Cartella di output
    )

    # Genera 1000 immagini bilanciate
    generator.generate_dataset(num_images=3000)

    # Oppure con distribuzione personalizzata
    # custom_dist = [
    #     {'ground': (2, 5), 'tower': (1, 3), 'air': (0, 2)},
    #     {'ground': (0, 2), 'tower': (3, 5), 'air': (1, 3)},
    # ]
    # generator.generate_dataset(num_images=1000, distribution=custom_dist)

Generate 100/3000 immagini
Generate 200/3000 immagini
Generate 300/3000 immagini
Generate 400/3000 immagini
Generate 500/3000 immagini
Generate 600/3000 immagini
Generate 700/3000 immagini
Generate 800/3000 immagini
Generate 900/3000 immagini
Generate 1000/3000 immagini
Generate 1100/3000 immagini
Generate 1200/3000 immagini
Generate 1300/3000 immagini
Generate 1400/3000 immagini
Generate 1500/3000 immagini
Generate 1600/3000 immagini
Generate 1700/3000 immagini
Generate 1800/3000 immagini
Generate 1900/3000 immagini
Generate 2000/3000 immagini
Generate 2100/3000 immagini
Generate 2200/3000 immagini
Generate 2300/3000 immagini
Generate 2400/3000 immagini
Generate 2500/3000 immagini
Generate 2600/3000 immagini
Generate 2700/3000 immagini
Generate 2800/3000 immagini
Generate 2900/3000 immagini
Generate 3000/3000 immagini
Dataset completato! 3000 immagini generate.
names: ['archer', 'barbarian', 'battleram', 'bomber', 'bombtower', 'cannon', 'electroSpirit', 'giant', 'goblin', 'goblinCage'

In [5]:
import json

# Il tuo mapping corretto
correct_map = {
    'archer': 0, 'barbarian': 1, 'battleram': 2, 'bomber': 3, 'cannon': 4,
    'electrospirit': 5, 'giant': 6, 'goblin': 7, 'goblinHut': 8, 'hogrider': 9,
    'infernotower': 10, 'knight': 11, 'minipekka': 12, 'mortar': 13, 'skeleton': 14,
    'spearGoblin': 15, 'superGoblin': 16, 'tombstone': 17, 'valk': 18, 'wizard': 19,
    'bombtower': 20, 'firespirit': 21, 'furnace': 22, 'goblinCage': 23, 'musketeer': 24,
    'bat': 25, 'flyingmachine': 26, 'megaminion': 27, 'minion': 28, 'skeletondragon': 29,
    'AllyHP': 30, 'EnemyHP': 31, 'enemyLevel': 32,
    'tower': 33
}


label_dir="../dataset/validation set/labels"
"""Corregge gli indici delle classi nei file .txt YOLO"""
from pathlib import Path


# Label Studio ordina alfabeticamente - crea mapping sbagliato
labelstudio_order = sorted(correct_map.keys())
labelstudio_map = {name: idx for idx, name in enumerate(labelstudio_order)}

# Mapping inverso: da indice_sbagliato -> nome -> indice_corretto
index_fix_map = {}
for name, correct_idx in correct_map.items():
    wrong_idx = labelstudio_map[name]
    index_fix_map[wrong_idx] = correct_idx

print("Mapping correzione:")
for wrong, correct in sorted(index_fix_map.items()):
    name = labelstudio_order[wrong]
    print(f"  {wrong} ({name}) -> {correct}")

# Correggi tutti i file
label_files = list(Path(label_dir).glob("*.txt"))
print(f"\nCorreggo {len(label_files)} file...")

for label_file in label_files:
    corrected_lines = []

    with open(label_file, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                wrong_class_id = int(parts[0])

                # Correggi l'indice
                if wrong_class_id in index_fix_map:
                    correct_class_id = index_fix_map[wrong_class_id]
                    parts[0] = str(correct_class_id)

                corrected_lines.append(' '.join(parts) + '\n')

    # Riscrivi file corretto
    with open(label_file, 'w') as f:
        f.writelines(corrected_lines)

print(f"✓ Corretto {len(label_files)} file!")


Mapping correzione:
  0 (AllyHP) -> 30
  1 (EnemyHP) -> 31
  2 (archer) -> 0
  3 (barbarian) -> 1
  4 (bat) -> 25
  5 (battleram) -> 2
  6 (bomber) -> 3
  7 (bombtower) -> 20
  8 (cannon) -> 4
  9 (electrospirit) -> 5
  10 (enemyLevel) -> 32
  11 (firespirit) -> 21
  12 (flyingmachine) -> 26
  13 (furnace) -> 22
  14 (giant) -> 6
  15 (goblin) -> 7
  16 (goblinCage) -> 23
  17 (goblinHut) -> 8
  18 (hogrider) -> 9
  19 (infernotower) -> 10
  20 (knight) -> 11
  21 (megaminion) -> 27
  22 (minion) -> 28
  23 (minipekka) -> 12
  24 (mortar) -> 13
  25 (musketeer) -> 24
  26 (skeleton) -> 14
  27 (skeletondragon) -> 29
  28 (spearGoblin) -> 15
  29 (superGoblin) -> 16
  30 (tombstone) -> 17
  31 (tower) -> 33
  32 (valk) -> 18
  33 (wizard) -> 19

Correggo 14 file...
✓ Corretto 14 file!
